In [0]:
from olist_lakehouse.io import read_managed_table, write_managed_table
from pyspark.sql.functions import col

In [0]:
dbutils.widgets.text("catalog", "")

In [0]:
catalog = dbutils.widgets.get("catalog")

## Order Items table
Enriched order items with sellers information.

**Table requirements:**

- Fields must be correctly typed

**Expected schema:**

| Field                  | Type   | Origin (bronze)           |
| -----------------------| -------| --------------------------|
| order_id               | STRING | olist_order_items_dataset |
| order_item_id          | INT    | olist_order_items_dataset |
| product_id             | STRING | olist_order_items_dataset |
| price                  | DOUBLE | olist_order_items_dataset |
| freight_value          | DOUBLE | olist_order_items_dataset |
| seller_id              | STRING | olist_sellers_dataset     |
| seller_city            | STRING | olist_sellers_dataset     |
| seller_state           | STRING | olist_sellers_dataset     |
| seller_zip_code_prefix | STRING | olist_sellers_dataset     |

In [0]:
order_items = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_order_items_dataset",
)
sellers = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_sellers_dataset",
)

In [0]:
order_items_join = order_items.join(sellers, on="seller_id", how="left")
order_items_transformed = order_items_join.select(
    "order_id",
    col("order_item_id").cast("int"),
    "product_id",
    col("price").cast("double"),
    col("freight_value").cast("double"),
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
)

In [0]:
write_managed_table(
    order_items_transformed,
    mode="overwrite",
    table_full_path=f"{catalog}.silver.order_items",
    options={"overwriteSchema": "true"}
)